In [4]:
import math

from qiskit_aer import AerSimulator
from qiskit_nature.second_q.operators import FermionicOp
from qiskit_nature.second_q.mappers import JordanWignerMapper

from qiskit import transpile
from qiskit.circuit import QuantumRegister, QuantumCircuit
from qiskit.circuit.library import PauliEvolutionGate
from qiskit.quantum_info import SparsePauliOp
from qiskit_aer import AerSimulator

import scipy as sp


![alt text](second%20quantized.png "second quantized")
From https://doi.org/10.1063/5.0150291

Terms on the fifth line are irrelevant, they are coulomb interactions invoving the classically treated nuclei.

# New try

In [5]:
import math
import numpy as np
import pyscf
from gbasis.parsers import parse_nwchem

elec_basis_dict = parse_nwchem("def2-SVP.nw")
nuc_basis_dict = parse_nwchem("DZSNB.nw")

dist_a = 0.7414
dist_bohr = dist_a * 1.8897259886

"""
for atom in basis_dict:
    print(f"Atom: {atom}")
    print(f"   Number of shells: {len(basis_dict[atom])}")
    for i, shell in enumerate(basis_dict[atom]):
        print(f"   Shell {i} has angular momentum {shell[0]}")
        print(f"   Shell {i} has exponents {shell[1]}")
        print(f"   Shell {i} has coefficients {shell[2].flatten()}")
"""

import numpy as np
import scipy as sp
from gbasis.parsers import parse_gbs, make_contractions
from gbasis.integrals.overlap import overlap_integral
from gbasis.integrals.overlap_asymm import overlap_integral_asymmetric
from gbasis.integrals.kinetic_energy import kinetic_energy_integral
from gbasis.integrals.electron_repulsion import electron_repulsion_integral

mass_proton = 1874.0  #mass of proton in atomic units (electron masses)

# Centers of electron orbitals (just the H atoms position)
elec_atoms = ["H", "H"]
elec_atcoords = np.array([[0.0, 0.0, 0.0], [0.0, 0.0, dist_bohr]])

# Centers of nuclear orbitals for protons
nuc_atoms = ["Q", "Q"]
nuc_atcoords = np.array([[0.0, 0.0, 0.0], [0.0, 0.0, dist_bohr]])


# Construct full molecular orbital  basis from atomic orbitals centered at the positions "atcoords"
elec_basis = make_contractions(elec_basis_dict, elec_atoms, elec_atcoords, coord_types="cartesian")
nuc_basis = make_contractions(nuc_basis_dict, nuc_atoms, nuc_atcoords, coord_types="cartesian")
full_basis = elec_basis + nuc_basis # Bases are tuples of contracted gaussian objects. NOTE that the overlap

In [6]:
# symmetric orthogonalization of AOs. Szabo sec 3.4.5
elec_overlap = overlap_integral(elec_basis)
elec_ortho = np.linalg.inv(sp.linalg.sqrtm(elec_overlap))  # Transform needed to get orthogonal MOs

# RHF calculation to get MOs that will be used for the quantum algorihtm
mol = pyscf.M(
    atom = 'H 0 0 0; H 0 0 0.7414',  # in Angstrom
    basis = 'def2-SVP',
    symmetry = True,
)
#=
hf = mol.HF().run()
elec_ortho = hf.mo_coeff.T # probably should get the integrals from pyscf too? in case they dont match gbasis. or figure a way to parse pyscf w gbasis.

nuc_overlap = overlap_integral(nuc_basis)
nuc_ortho = np.linalg.inv(sp.linalg.sqrtm(nuc_overlap))  # Transform needed to get orthogonal MOs

# basis function count
elec_bc = elec_overlap.shape[0]
nuc_bc = nuc_overlap.shape[0]


full_ortho = np.block([[elec_ortho, np.zeros((elec_bc, nuc_bc))],
                       [np.zeros((nuc_bc, elec_bc)), nuc_ortho]])

full_overlap = overlap_integral(full_basis, transform=full_ortho)
print(np.matrix.round(full_overlap, 7))
# Note the "non-zero" overlap between nuclear and electronic orbitals. They actually are orthogonal in the Hilbert space though.


converged SCF energy = -1.12890637848863
[[ 1.000000e+00  0.000000e+00 -0.000000e+00  0.000000e+00  0.000000e+00
   0.000000e+00 -0.000000e+00  0.000000e+00  0.000000e+00 -0.000000e+00
  -6.991900e-03  1.972916e-01 -6.991900e-03  1.972916e-01]
 [ 0.000000e+00  9.999999e-01 -0.000000e+00 -0.000000e+00  0.000000e+00
   0.000000e+00 -0.000000e+00  0.000000e+00  0.000000e+00 -1.000000e-07
  -2.386900e-03  7.021080e-02  2.386900e-03 -7.021080e-02]
 [-0.000000e+00 -0.000000e+00  1.000000e+00  0.000000e+00  0.000000e+00
   0.000000e+00 -0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00
  -2.851100e-03 -1.830728e-01 -2.851100e-03 -1.830728e-01]
 [ 0.000000e+00 -0.000000e+00 -0.000000e+00  1.000000e+00  0.000000e+00
   0.000000e+00 -0.000000e+00  0.000000e+00  0.000000e+00  2.000000e-07
  -2.323700e-03 -2.507715e-01  2.323700e-03  2.507715e-01]
 [ 0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00  1.000000e+00
   0.000000e+00  0.000000e+00  0.000000e+00 -0.000000e+00  0.000000e+00
   

In [41]:
elec_ke = kinetic_energy_integral(elec_basis, transform=elec_ortho)
print(elec_ke)

nuc_ke = kinetic_energy_integral(nuc_basis, transform=nuc_ortho) / mass_proton
print(nuc_ke)

ee_coulomb =electron_repulsion_integral(elec_basis, transform=elec_ortho)

nn_coulomb = electron_repulsion_integral(nuc_basis, transform=nuc_ortho)

coulomb = electron_repulsion_integral(full_basis, transform=full_ortho)

en_coulomb = coulomb[0:elec_bc, elec_bc:elec_bc+nuc_bc, 0:elec_bc, elec_bc:elec_bc+nuc_bc]


[[ 5.48325427e-01  2.18056191e-17 -5.23593808e-01  6.61658167e-17
   0.00000000e+00  0.00000000e+00  6.82018154e-02  0.00000000e+00
   0.00000000e+00  1.12118225e-16]
 [ 4.71471345e-17  3.42058998e-01 -9.94370956e-17 -2.78557095e-01
   0.00000000e+00  0.00000000e+00 -1.00532082e-16  0.00000000e+00
   0.00000000e+00  1.37513852e-01]
 [-5.23593808e-01 -4.99228076e-17  8.81740748e-01 -9.79044973e-17
   0.00000000e+00  0.00000000e+00 -6.75634763e-02  0.00000000e+00
   0.00000000e+00 -3.44630128e-16]
 [ 1.00932492e-17 -2.78557095e-01  1.81969967e-17  1.72166758e+00
   0.00000000e+00  0.00000000e+00 -2.36680833e-17  0.00000000e+00
   0.00000000e+00 -5.52919296e-01]
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00
   1.80326384e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00
   5.89586070e-17  0.00000000e+00]
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00
   0.00000000e+00  1.80326384e+00  0.00000000e+00  5.89586070e-17
   0.00000000e+00  0.00000000e+00

In [43]:
print(en_coulomb[0,0,0,0])
print(coulomb[0,elec_bc,0,elec_bc])

0.8911602130483977
0.8911602130483977


# Physicist notation integral
![](physnot.png)
this is coulomb[a,b,c,d]

In [8]:
# mo count
elec_moc = (elec_bc-7) # truncate 4 basis states
nuc_moc = nuc_bc

elec_modes = elec_moc * 2
nuc_modes = nuc_moc * 2
total_modes = elec_modes + nuc_modes

# KE of electron and nuclei in terms of annihilation/creation ops
elec_KE_fermion_op = FermionicOp({}, num_spin_orbitals=elec_modes)
nuc_KE_fermion_op = FermionicOp({}, num_spin_orbitals=nuc_modes)

# SparsePauliOp identity for  qubits representing the electronic modes
elec_pauli_identity = SparsePauliOp("I"*elec_modes)
nuc_pauli_identity = SparsePauliOp("I"*nuc_modes)

mapper = JordanWignerMapper()



for i in range(elec_moc):
    for j in range(elec_moc):
        # Only include terms with the same spin: 2i, 2j and 2i+1, 2j+1 (alpha and beta orbitals respectively)
        elec_KE_fermion_op += FermionicOp({f"+_{2*i} -_{2*j}": elec_ke[i, j]}, num_spin_orbitals=elec_modes)
        elec_KE_fermion_op += FermionicOp({f"+_{2*i+ 1} -_{2*j + 1}": elec_ke[i, j]}, num_spin_orbitals=elec_modes)

for i in range(nuc_moc):
    for j in range(nuc_moc):
        # Only include terms with the same spin: 2i, 2j and 2i+1, 2j+1 (alpha and beta orbitals respectively)
        nuc_KE_fermion_op += FermionicOp({f"+_{2*i} -_{2*j}": nuc_ke[i, j]}, num_spin_orbitals=nuc_modes)
        nuc_KE_fermion_op += FermionicOp({f"+_{2*i + 1} -_{2*j + 1}": nuc_ke[i, j]}, num_spin_orbitals=nuc_modes)


#print(elec_KE_fermion_op)
#print(nuc_KE_fermion_op)

elec_KE_pauli_op = mapper.map(elec_KE_fermion_op)
nuc_KE_pauli_op = mapper.map(nuc_KE_fermion_op)

KE_pauli_op = (elec_KE_pauli_op  ^ nuc_pauli_identity) + (elec_pauli_identity ^ nuc_KE_pauli_op)



In [9]:
elec_elec_coulomb_fermion_op = FermionicOp({}, num_spin_orbitals=elec_modes)
nuc_nuc_coulomb_fermion_op = FermionicOp({}, num_spin_orbitals=nuc_modes)

for i in range(elec_moc):
    for j in range(elec_moc):
        for k in range(elec_moc):
            for l in range(elec_moc):
                for spin1 in range(2):
                    for spin2 in range(2):
                        elec_elec_coulomb_fermion_op += FermionicOp({
                            f"+_{2*i + spin1} +_{2*j + spin2} -_{2*k + spin2} -_{2*l + spin1}": 0.5 * coulomb[i, j, l, k],
                        }, num_spin_orbitals=elec_modes)

for i in range(nuc_moc):
    for j in range(nuc_moc):
        for k in range(nuc_moc):
            for l in range(nuc_moc):
                for spin1 in range(2):
                    for spin2 in range(2):
                        nuc_nuc_coulomb_fermion_op += FermionicOp({
                            f"+_{2*i + spin1} +_{2*j + spin2} -_{2*k + spin2} -_{2*l + spin1}": 0.5 * coulomb[i + elec_moc, j + elec_moc, l + elec_moc, k + elec_moc],
                        }, num_spin_orbitals=nuc_modes)

# cant combine elec and nuc fermion ops or else the jordan wigner mapper will think theyre anticommuting (they commute. indistinguishable!)
# so i will have to convert the 2 one body ops (one nuclear and one electronic) to qubit via JWT, SEPARATELY, then combine them (tensor product)

elec_elec_coulomb_pauli_op = mapper.map(elec_elec_coulomb_fermion_op)
nuc_nuc_coulomb_pauli_op = mapper.map(nuc_nuc_coulomb_fermion_op)



"""
for i in range(elec_moc):
    for j in range(nuc_moc):
        for k in range(elec_moc):
            for l in range(nuc_moc):
                for spin1 in range(2):
                    for spin2 in range(2):
                        elec_fermion_op = FermionicOp({f"+_{2*i + spin1} -_{2*k + spin1}": 1,}, num_spin_orbitals=elec_modes)
                        nuc_fermion_op = FermionicOp({f"+_{2*j + spin2} -_{2*l + spin2}": 1,}, num_spin_orbitals=nuc_modes)

                        combine_pauli_op = coulomb[i, j + elec_moc, k, l + elec_moc] * (mapper.map(elec_fermion_op) ^ mapper.map(nuc_fermion_op))

                        print(combine_pauli_op)

                        elec_nuc_coulomb_pauli_op += combine_pauli_op
                        print(np.allclose(combine_pauli_op.coeffs.imag, 0), i, j + elec_moc, k, l + elec_moc)

elec_nuc_coulomb_pauli_op = -elec_nuc_coulomb_pauli_op
"""



'\nfor i in range(elec_moc):\n    for j in range(nuc_moc):\n        for k in range(elec_moc):\n            for l in range(nuc_moc):\n                for spin1 in range(2):\n                    for spin2 in range(2):\n                        elec_fermion_op = FermionicOp({f"+_{2*i + spin1} -_{2*k + spin1}": 1,}, num_spin_orbitals=elec_modes)\n                        nuc_fermion_op = FermionicOp({f"+_{2*j + spin2} -_{2*l + spin2}": 1,}, num_spin_orbitals=nuc_modes)\n\n                        combine_pauli_op = coulomb[i, j + elec_moc, k, l + elec_moc] * (mapper.map(elec_fermion_op) ^ mapper.map(nuc_fermion_op))\n\n                        print(combine_pauli_op)\n\n                        elec_nuc_coulomb_pauli_op += combine_pauli_op\n                        print(np.allclose(combine_pauli_op.coeffs.imag, 0), i, j + elec_moc, k, l + elec_moc)\n\nelec_nuc_coulomb_pauli_op = -elec_nuc_coulomb_pauli_op\n'

In [64]:
elec_nuc_coulomb_pauli_op = 0

for i in range(elec_moc):
    for j in range(nuc_moc):
        for k in range(i + 1):
            for l in range(j + 1):
                for spin1 in range(2):
                    for spin2 in range(2):
                        if i != k:
                            elec_fermion_op = FermionicOp({f"+_{2*i + spin1} -_{2*k + spin1}": 1, f"+_{2*k + spin1} -_{2*i + spin1}": 1,}, num_spin_orbitals=elec_modes)
                        else:
                            elec_fermion_op = FermionicOp({f"+_{2*i + spin1} -_{2*k + spin1}": 1,}, num_spin_orbitals=elec_modes)

                        if j != l:
                            nuc_fermion_op = FermionicOp({f"+_{2*j + spin2} -_{2*l + spin2}": 1, f"+_{2*l + spin2} -_{2*j + spin2}" : 1,}, num_spin_orbitals=nuc_modes)
                        else:
                            nuc_fermion_op = FermionicOp({f"+_{2*j + spin2} -_{2*l + spin2}": 1}, num_spin_orbitals=nuc_modes)
                        combine_pauli_op = coulomb[i, j + elec_moc, k, l + elec_moc] * (mapper.map(elec_fermion_op) ^ mapper.map(nuc_fermion_op))
                        #print(elec_fermion_op)
                        #print(nuc_fermion_op)
                        #print(combine_pauli_op)

                        elec_nuc_coulomb_pauli_op += combine_pauli_op
                        #print(np.allclose(combine_pauli_op.coeffs.imag, 0), i, j + elec_moc, k, l + elec_moc)

elec_nuc_coulomb_pauli_op = elec_nuc_coulomb_pauli_op


In [ ]:
print(coulomb[1,13,1,10])
print(coulomb[0,10,0,6])

In [ ]:
hamiltonian = KE_pauli_op + (elec_elec_coulomb_pauli_op  ^ nuc_pauli_identity) + (elec_pauli_identity ^ nuc_nuc_coulomb_pauli_op) #+ elec_nuc_coulomb_pauli_op

In [ ]:
from qiskit.quantum_info import Statevector

a = Statevector.from_label(valid_states[0])
b = hamiltonian @ a

In [ ]:
from qiskit.synthesis import LieTrotter

qubits = total_modes

qc = QuantumCircuit(qubits)

evolution_gate = PauliEvolutionGate(hamiltonian, time=0.01, synthesis=LieTrotter(reps=10))

qc.append(evolution_gate, range(qubits))



In [ ]:
elec_number_fermion_ops = [FermionicOp({f"+_{i} -_{i}": 1}, num_spin_orbitals=elec_modes) for i in range(elec_modes)]
elec_number_fermion_op = sum(elec_number_fermion_ops)

elec_number_pauli_ops = [mapper.map(fermion_op) for fermion_op in elec_number_fermion_ops]
elec_number_pauli_op = mapper.map(elec_number_fermion_op)

nuc_number_fermion_ops = [FermionicOp({f"+_{i} -_{i}": 1}, num_spin_orbitals=nuc_modes) for i in range(nuc_modes)]
nuc_number_fermion_op = sum(nuc_number_fermion_ops)

nuc_number_pauli_ops = [mapper.map(fermion_op) for fermion_op in nuc_number_fermion_ops]
nuc_number_pauli_op = mapper.map(nuc_number_fermion_op)

elec_number_pauli_op_full = elec_number_pauli_op ^ nuc_pauli_identity
nuc_number_pauli_op_full = elec_pauli_identity ^ nuc_number_pauli_op


In [ ]:
print(elec_number_pauli_ops[0])

In [ ]:
a = SparsePauliOp(['IIZ']).to_matrix()
b = SparsePauliOp(['IZI']).to_matrix()
c = SparsePauliOp(['ZII']).to_matrix()

In [ ]:
b

In [ ]:
hamiltonian_sparse = hamiltonian.to_matrix(sparse=True)
#elec_number_sparse = elec_number_pauli_op_full.to_matrix(sparse=True)
#nuc_number_sparse = nuc_number_pauli_op_full.to_matrix(sparse=True)

In [ ]:
hamiltonian_sparse.data[np.abs(hamiltonian_sparse.data) < 1e-4] = 0

hamiltonian_sparse.eliminate_zeros()

In [ ]:
eigenvalues, eigenvectors = sp.sparse.linalg.eigs(elec_number_sparse + nuc_number_sparse, which='LM', k=16)

In [11]:
import itertools

num_electrons = 2
num_protons = 2

valid_states = [] #binary strings in occ number basis corresponding to states with 2 electorns and 2 protons
valid_states_indices = [] #decimal version of these binary strings

for elec_indices in itertools.combinations(range(elec_modes), num_electrons):
    elec_string = ['0'] * elec_modes
    for i in elec_indices:
        elec_string[i] = '1'
    for nuc_indices in itertools.combinations(range(nuc_modes), num_protons):
        nuc_string = ['0'] * nuc_modes
        for i in nuc_indices:
                nuc_string[i] = '1'

        total_string = "".join(elec_string + nuc_string)

        valid_states.append(total_string)
        valid_states_indices.append(int(total_string, 2))

In [28]:
import itertools

num_electrons = 2
num_protons = 2

evalid_states = [] #binary strings in occ number basis corresponding to states with 2 electorns and 2 protons
evalid_states_indices = [] #decimal version of these binary strings

for elec_indices in itertools.combinations(range(elec_modes), num_electrons):
    elec_string = ['0'] * elec_modes
    for i in elec_indices:
        elec_string[i] = '1'

    total_string = "".join(elec_string)
    evalid_states.append(total_string)
    evalid_states_indices.append(int(total_string, 2))

print(evalid_states)
print(evalid_states_indices)

['110000', '101000', '100100', '100010', '100001', '011000', '010100', '010010', '010001', '001100', '001010', '001001', '000110', '000101', '000011']
[48, 40, 36, 34, 33, 24, 20, 18, 17, 12, 10, 9, 6, 5, 3]


In [27]:
import itertools

num_electrons = 2
num_protons = 2

nvalid_states = [] #binary strings in occ number basis corresponding to states with 2 electorns and 2 protons
nvalid_states_indices = [] #decimal version of these binary strings

for indices in itertools.combinations(range(nuc_modes), num_protons):
    string = ['0'] * nuc_modes
    for i in indices:
        string[i] = '1'

    total_string = "".join(string)
    nvalid_states.append(total_string)
    nvalid_states_indices.append(int(total_string, 2))

print(nvalid_states)
print(nvalid_states_indices)

['11000000', '10100000', '10010000', '10001000', '10000100', '10000010', '10000001', '01100000', '01010000', '01001000', '01000100', '01000010', '01000001', '00110000', '00101000', '00100100', '00100010', '00100001', '00011000', '00010100', '00010010', '00010001', '00001100', '00001010', '00001001', '00000110', '00000101', '00000011']
[192, 160, 144, 136, 132, 130, 129, 96, 80, 72, 68, 66, 65, 48, 40, 36, 34, 33, 24, 20, 18, 17, 12, 10, 9, 6, 5, 3]


In [29]:
eKE_mtx = elec_KE_pauli_op.to_matrix()
eCmb_mtx = elec_elec_coulomb_pauli_op.to_matrix()
neCmb_mtx = elec_nuc_coulomb_pauli_op.to_matrix()

# Calculating matrix elements between determinants/permanents
This will allow me to directly calculate correlation functions for an N-particle subspace. It will give me the exact result that the quantum algorithm would aim to replicate

In [17]:
# class that wraps an array of one-body SPATIAL ORBITAL integrals into SPIN ORBITAL integrals by returning integrals that don't match spin to 0
class IntegralSpinWrapper:
    def __init__(self, data, spin):
        self.data = data
        self.spin = spin

    def __getitem__(self, index):
        if len(index) == 2:
            if (index[0] % self.spin == index[1] % self.spin):
                return self.data[int(index[0]/self.spin), int(index[1]/self.spin)]

            # if spins don't match the integral will be 0
            else:
                return 0

        elif len(index) == 4:
            if (index[0] % self.spin == index[2] % self.spin and index[1] % self.spin == index[3] % self.spin):
                return self.data[int(index[0]/self.spin), int(index[1]/self.spin), int(index[2]/self.spin), int(index[3]/self.spin)]

            # if spins don't match the integral will be 0
            else:
                return 0
        else:
            raise IndexError


In [65]:
wrapped_elec_ke = IntegralSpinWrapper(elec_ke, 2)
wrapped_nuc_ke = IntegralSpinWrapper(nuc_ke, 2)
wrapped_elec_coulomb = IntegralSpinWrapper(ee_coulomb, 2)
wrapped_nuc_coulomb = IntegralSpinWrapper(nn_coulomb, 2)
wrapped_en_coulomb = IntegralSpinWrapper(en_coulomb, 2)

In [19]:
# one-body matrix element for determinants
# bra and ket are binary strings for which states are occupied
# itg are the integrals
#
# see szabo pg. 70
#
# example for spin 1/2
#
def det_1_mtx_el(bra, ket, itg):
    st_count = len(bra) # number of states

    common = [st_count - 1 - i for i in range(st_count) if bra[i] == '1' and ket[i] == '1'][::-1]
    diff = [st_count - 1 - i for i in range(st_count) if bra[i] != ket[i]][::-1]  # note st_count - 1 - i so we get indices for the integrals, not indices in the string (they are reverse from each other)

    diff_bra = [st_count - 1 - i for i in range(st_count) if bra[i] == '1' and ket[i] == '0'][::-1] # states that are only in the bra determinant. we reverse because of the reverse ordering
    diff_ket = [st_count - 1 - i for i in range(st_count) if bra[i] == '0' and ket[i] == '1'][::-1] # states that are only in the ket


    # case 1: same determinant
    if len(diff) == 0:
        return sum([itg[i, i] for i in common])

    # case 2: differs by one state
    elif len(diff) == 2:
        # number of permutations required to align the states = number of states in COMMON between the two unique states
        perms = len([i for i in common if diff[0] < i < diff[1]])

        # parity factor after aligning the states in the 2 determinants
        parity = 1 if perms % 2 == 0 else -1

        return parity * itg[diff_bra[0], diff_ket[0]]

    # case 3: differs by more than one state
    else:
        return 0

est1 = '010001'
est2 = '010100'

print(det_1_mtx_el(est1, est2, wrapped_elec_ke))
print(eKE_mtx[int(est1, 2), int(est2, 2)])


2.1805619127679654e-17
0j


In [44]:
# notes: first, the states are labelled from rightmost bits to left.
# the state 001110 is a slater determinant formed from states 1, 2, and 3 IN THAT ORDER. or |123> for short
#          ^543210^
#
# szabo's formula assumes that the determinants match for every state except 1 or 2. thus we need to apply the correct factor +/-1
# depending on if it takes an even or odd number of permutations to align the determinant
#
# ex: 00111001:         determinant |0345>
#     01101010:                     |1356>
#
# but to use szabo's rule we need   |0345>
#                                   |1365> or something like this so that the states in common are matched
#
#

# theres some sign issue here.

def det_2_mtx_el(bra, ket, itg):
    st_count = len(bra) # number of states

    common = [st_count - 1 - i for i in range(st_count) if bra[i] == '1' and ket[i] == '1'][::-1]
    diff = [st_count - 1 - i for i in range(st_count) if bra[i] != ket[i]][::-1]  # note st_count - 1 - i so we get indices for the integrals, not indices in the string (they are reverse from each other)

    diff_bra = [st_count - 1 - i for i in range(st_count) if bra[i] == '1' and ket[i] == '0'][::-1] # states that are only in the bra determinant. we reverse because of the reverse ordering
    diff_ket = [st_count - 1 - i for i in range(st_count) if bra[i] == '0' and ket[i] == '1'][::-1] # states that are only in the ket

    # case 1: same determinant
    if len(diff) == 0:
        # szabo has [ii|jj] - [ij|ji] this is CHEMISTS notation. our integrals are in physicists
        # physicists notation becomes <ij|ij> - <ij|ji>
        return 0.5*sum([sum([itg[i,j,i,j]-itg[i,j,j,i] for i in common]) for j in common])

    # case 2: differs by one state
    elif len(diff) == 2:
        # number of permutations required to align the states = number of states in COMMON between the two unique states
        perms = len([i for i in common if diff[0] < i < diff[1]])

        # parity factor after aligning the states in the 2 determinants
        parity = 1 if perms % 2 == 0 else -1

        return parity * sum([itg[diff_bra[0], i, diff_ket[0], i] - itg[diff_bra[0], i, i, diff_ket[0]] for i in common])

    # case 3: differs by two states
    elif len(diff) == 4:
        perms = len([i for i in common if diff[0] < i < diff[1] or diff[2] < i < diff[3]])
        parity = 1 if perms % 2 == 0 else -1
        #print('chung')
        #print(diff_bra, diff_ket)
        #print(bra, ket)
        #print(itg[diff_bra[0], diff_bra[1], diff_ket[0], diff_ket[1]], itg[diff_bra[0], diff_bra[1], diff_ket[1], diff_ket[0]])
        return parity * (itg[diff_bra[0], diff_bra[1], diff_ket[0], diff_ket[1]] - itg[diff_bra[0], diff_bra[1], diff_ket[1], diff_ket[0]])

    else:
        return 0


In [45]:
# two body matrix element for two DISTINGUISHABLE particle types
# case 1: sum_i,j [ii|jj] if the determinants are the exact same for both particle types
# case 2: sum_j of [ab|jj] if the determinants are the same for particle type 2, but has a in the bra determinant and b in the ket determinant for particle type 1 (differs by only one state)
# case 3: sum_i of [jj|ab} if ...
# case 4: [ab|cd] if
def dist_2_mtx_el(bra1, ket1, bra2, ket2, itg):
    st_count1 = len(bra1) # number of states of particle type 1
    st_count2 = len(bra2) # number of states of particle type 2

    common1 = [st_count1 - 1 - i for i in range(st_count1) if bra1[i] == '1' and ket1[i] == '1'][::-1]
    diff1 = [st_count1 - 1 - i for i in range(st_count1) if bra1[i] != ket1[i]][::-1]  # note st_count - 1 - i so we get indices for the integrals, not indices in the string (they are reverse from each other)

    common2 = [st_count2 - 1 - i for i in range(st_count2) if bra2[i] == '1' and ket2[i] == '1'][::-1]
    diff2 = [st_count2 - 1 - i for i in range(st_count2) if bra2[i] != ket2[i]][::-1]

    diff1_bra = [st_count1 - 1 - i for i in range(st_count1) if bra1[i] == '1' and ket1[i] == '0'][::-1] # states that are only in the bra determinant. we reverse because of the reverse ordering
    diff1_ket = [st_count1 - 1 - i for i in range(st_count1) if bra1[i] == '0' and ket1[i] == '1'][::-1] # states that are only in the ket

    diff2_bra = [st_count2 - 1 - i for i in range(st_count2) if bra2[i] == '1' and ket2[i] == '0'][::-1] # states that are only in the bra determinant. we reverse because of the reverse ordering
    diff2_ket = [st_count2 - 1 - i for i in range(st_count2) if bra2[i] == '0' and ket2[i] == '1'][::-1] # states that are only in the ket

    # case 1: same determinant for both particles
    if len(diff1) == 0 and len(diff2) == 0:
        return sum([sum([itg[i,j,i,j] for i in common1]) for j in common2])

    # case 2: differs by one state for particle 1, same det for particle 2
    elif len(diff1) == 2 and len(diff2) == 0:
        # number of permutations required to align the states = number of states in COMMON between the two unique states
        perms1 = len([i for i in common1 if diff1[0] < i < diff1[1]])

        # parity factor after aligning the states in the 2 determinants
        parity1 = 1 if perms1 % 2 == 0 else -1

        return parity1 * sum([itg[diff1_bra[0], i, diff1_ket[0], i] for i in common2])

    # case 3: differs by one state for particle 1, same det for particle 2
    elif len(diff1) == 2 and len(diff2) == 0:
        # number of permutations required to align the states = number of states in COMMON between the two unique states
        perms2 = len([i for i in common2 if diff2[0] < i < diff2[1]])

        # parity factor after aligning the states in the 2 determinants
        parity2 = 1 if perms2 % 2 == 0 else -1

        return parity2 * sum([itg[i, diff2_bra[0], i, diff2_ket[0]] for i in common1])

    # case 4: differs by one state for particles 1 and 2
    elif len(diff1) == 2 and len(diff2) == 2:
        perms1 = len([i for i in common1 if diff1[0] < i < diff1[1]])
        perms2 = len([i for i in common2 if diff2[0] < i < diff2[1]])

        # parity factor after aligning the states in the 2 determinants
        parity = 1 if (perms1 + perms2) % 2 == 0 else -1

        return parity * itg[diff1_bra[0], diff2_bra[0], diff1_ket[0], diff2_ket[0]]

    else:
        return 0

In [22]:
for est1 in evalid_states:
        a = det_1_mtx_el(est1, est2, wrapped_elec_ke)
        b = eKE_mtx[int(est1, 2), int(est2, 2)]
        c = abs(a - b)
        print (c < 1e-5)
        #print(a,b)
        if c >= 1e-5:
            print(est1, est2)
            print(a, b)


True
True
True
True
True
True
True
True
True
True
True
True
True
True
True


In [72]:
for est1 in evalid_states:
    for est2 in evalid_states:
        a = det_2_mtx_el(est1, est2, wrapped_elec_coulomb)
        b = eCmb_mtx[int(est1, 2), int(est2, 2)]
        c = abs(a - b)
        print (c < 1e-5)
        #print(a,b)
        if c >= 1e-5:
            print(est1, est2)
            print(a, b)


True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True


In [46]:
for est1 in evalid_states:
    for nst1 in nvalid_states:
        for est2 in evalid_states:
            for nst2 in nvalid_states:
                a = dist_2_mtx_el(est1, est2, nst1, nst2, wrapped_en_coulomb)
                b = neCmb_mtx[int(est1 + nst1, 2), int(est2 + nst2, 2)]
                c = abs(a - b)
                print(a,b)
                print (c < 1e-5)
                #print(a,b)
                if c >= 1e-5:
                    print(est1 + nst1, est2 + nst2)
                    print(a, b)


0.5791344031938374 (-1.7682178537402768+0j)
False
11000011000000 11000011000000
0.5791344031938374 (-1.7682178537402768+0j)
0 0j
True
0 0j
True
0 0j
True
0 0j
True
0 0j
True
0 (-9.71000529308181e-16+0j)
True
0 0j
True
0 0j
True
0 0j
True
0 0j
True
0 (9.71000529308181e-16+0j)
True
0 0j
True
0 0j
True
0 0j
True
0 0j
True
0 0j
True
0 0j
True
0 0j
True
0 0j
True
0 0j
True
0 0j
True
0 0j
True
0 0j
True
0 0j
True
0 0j
True
0 0j
True
0 0j
True
0 0j
True
0 0j
True
0 0j
True
0 0j
True
0 0j
True
0 0j
True
0 0j
True
0 0j
True
0 0j
True
0 0j
True
0 0j
True
0 0j
True
0 0j
True
0 0j
True
0 0j
True
0 0j
True
0 0j
True
0 0j
True
0 0j
True
0 0j
True
0 0j
True
0 0j
True
0 0j
True
0 0j
True
0 0j
True
0 0j
True
0 0j
True
0 0j
True
0.014936081154219143 (-9.043907208303406e-16+0j)
False
11000011000000 10010011000000
0.014936081154219143 (-9.043907208303406e-16+0j)
0 0j
True
-0.00551044296105025 0j
False
11000011000000 10010010010000
-0.00551044296105025 0j
0 0j
True
-0.002064629917264331 0j
False
1100001100

In [66]:
st1 = valid_states[0]
st2 = valid_states[0]

print(st1, st2)
a = neCmb_mtx[int(st1, 2), int(st2, 2)]
print(a)
print(en_coulomb[2,3,2,3]*4)
print(wrapped_en_coulomb[4, 6, 4, 6])
print(wrapped_en_coulomb[5, 6, 5, 6])
print(dist_2_mtx_el('110000', '110000', '11000000', '11000000', wrapped_en_coulomb))

11000011000000 11000011000000
(-1.7682178537402768+0j)
2.345251729891047
0.5863129324727617
0.5863129324727617
2.345251729891047


In [67]:
def h_mtx_el(est1, nst1, est2, nst2):
    e_ke = det_1_mtx_el(est1, est2, wrapped_elec_ke)
    n_ke = det_1_mtx_el(nst1, nst2, wrapped_nuc_ke)

    ee_cmb = det_2_mtx_el(est1, est2, wrapped_elec_coulomb)
    nn_cmb = det_2_mtx_el(nst1, nst2, wrapped_nuc_coulomb)
    en_cmb = dist_2_mtx_el(est1, est2, nst1, nst2, wrapped_en_coulomb)

    return e_ke + n_ke + ee_cmb + nn_cmb - en_cmb


In [73]:
print(h_mtx_el('010000', '10000000', '100000', '10000000'))

0.0162033760655457


In [83]:

e_stc = len(evalid_states) # number of electronic states with the correct partilce number
n_stc = len(nvalid_states) # ""        nuclear
stc = e_stc * n_stc

h = []

for st_bra in valid_states:
    h_row = []
    for st_ket in valid_states:
        h_row.append(h_mtx_el(st_bra[:elec_modes], st_bra[elec_modes:], st_ket[:elec_modes], st_ket[elec_modes:]))
    h.append(h_row)



In [89]:
print(h[2][4])

print(valid_states[2])
print(valid_states[4])

print(h_mtx_el('110000', '10010000', '110000', '10000100'))

2.1231459908282333
11000010010000
11000010000100
2.1231459908282333


In [49]:
index = (4, 9)
spin = 3
print(index[0] % spin == index[1] % spin)
print(int(7/8))


False
0


In [ ]:
all_states_indices = list(range(2**(elec_modes + nuc_modes)))
invalid_states_indices = [index for index in all_states_indices if index not in valid_states_indices]
print(invalid_states_indices)

In [74]:
print(valid_states[0])

11000011000000


In [ ]:
print(hamiltonian_sparse[np.ix_(valid_states_indices, valid_states_indices)])


hamiltonian_valid = hamiltonian_sparse[np.ix_(valid_states_indices, valid_states_indices)]

eigenvalues, eigenvectors = sp.sparse.linalg.eigsh(hamiltonian_valid, which='SM', k=10)

In [ ]:
print(eigenvalues)

In [ ]:
hamiltonian_sparse_matrix.data[np.abs(hamiltonian_sparse_matrix.data) < 1e-3] = 0

hamiltonian_sparse_matrix.eliminate_zeros()
#print(hamiltonian_sparse_matrix)
eigenvalues, eigenvectors = sp.sparse.linalg.eigs(hamiltonian_sparse_matrix, which='SM', k=1, sigma=-1.06)

In [ ]:
print(eigenvalues)

In [ ]:
print(np.matrix.round(eigenvectors, 2)[:, 1])

In [ ]:

#test = overlap_integral(nuc_basis, transform=nuc_ortho)
#test2 = overlap_integral(elec_basis, transform=elec_ortho)
#print(test2)
# Construct representation of CAR (fermionic creation/annihilation operators)
ident = [[1, 0], [0, 1]]
pauli_x = [[0, 1], [1, 0]]
pauli_y = [[0, -1j], [1j, 0]]
pauli_z = [[1, 0], [0, -1]]

modes = 2

# create representation of system of N distinguishable spins
pauli_xs = []
pauli_ys = []
pauli_zs = []

spin_raise = []
spin_lower = []

create = []
annihilate = []

for i in range(modes):
    x = y = z = [1]
    for j in range(modes):
        if i == j:
            x = np.kron(x, pauli_x)
            y = np.kron(y, pauli_y)
            z = np.kron(z, pauli_z)
        else:
            x = np.kron(x, ident)
            y = np.kron(y, ident)
            z = np.kron(z, ident)

    pauli_xs.append(x)
    pauli_ys.append(y)
    pauli_zs.append(z)

    spin_raise.append(0.5 * (x + 1j * y))
    spin_lower.append(0.5 * (x - 1j * y))

# construct fermionic creation/annihilation operators via JWT
for i in range(modes):
    sum = np.zeros([2 ** modes, 2 ** modes])
    for j in range(i):
        sum = sum + (spin_raise[j] @ spin_lower[j])

    create_op = sp.linalg.expm(1j * math.pi * sum) @ spin_raise[i]
    annihilate_op = sp.linalg.expm(-1j * math.pi * sum) @ spin_lower[i]

    annihilate.append(annihilate_op)
    create.append(create_op)

for i in range(modes):
    for j in range(modes):
        #commutator = annihilate[i] @ annihilate[j] + annihilate[j] @ annihilate[i]
        commutator = create[i] @ create[j] + create[j] @ create[i]
        #commutator = annihilate[i] @ create[j] + create[j] @ annihilate[i]
        #print(commutator)

        if (np.allclose(commutator, np.eye(2 ** modes))):
            print(1)
        elif (np.allclose(commutator, np.zeros([2 ** modes, 2 ** modes]))):
            print(0)
        else:
            print("?")

print("bruh")



In [ ]:
print(type(elec_basis))z